In [1]:
import json

import numpy as np


def compute_statistics(all_logs: list) -> dict:
    """Barcha loglardan o'rtacha qiymat va standart og'ishlarni hisoblaydi."""
    bytes_list = [log.get('data_transferred_bytes', 0) for log in all_logs]
    failed_list = [0 if log.get('status_code', 200) == 200 else 1 for log in all_logs]

    mean_bytes = float(np.mean(bytes_list))
    std_bytes = float(np.std(bytes_list))
    mean_failed = float(np.mean(failed_list))
    std_failed = float(np.std(failed_list))

    return {
        'mean_bytes': mean_bytes,
        'std_bytes': std_bytes if std_bytes > 0 else 1.0,
        'mean_failed': mean_failed,
        'std_failed': std_failed if std_failed > 0 else 1.0,
    }


def analyze_log_item(log_data: dict, stats: dict) -> dict:
    """1 dona log uchun risk hisoblash funksiyasi."""
    mean_bytes = stats['mean_bytes']
    std_bytes = stats['std_bytes']
    mean_failed = stats['mean_failed']
    std_failed = stats['std_failed']

    bytes_sent = log_data.get('data_transferred_bytes', 0)
    status_code = log_data.get('status_code', 200)
    country = log_data.get('source_country', 'UZ')
    hour = log_data.get('hour', 12)

    # Z-Score
    z_bytes = (bytes_sent - mean_bytes) / (std_bytes + 1e-6)
    is_failed = 1 if status_code != 200 else 0
    z_failed = (is_failed - mean_failed) / (std_failed + 1e-6)

    # Risk qoidalari
    geo_risk = 2.0 if country != 'UZ' else 0.0
    time_risk = 1.5 if (0 <= hour < 6) else 0.0

    # Risk Score
    risk_score = np.sqrt(z_bytes**2 + z_failed**2) + geo_risk + time_risk
    is_anomaly = float(risk_score) > 2.0

    return {
        'user_id': log_data.get('user_id'),
        'risk_score': round(float(risk_score), 2),
        'is_anomaly': is_anomaly,
        'log': log_data,
    }


def process_test_file(file_path: str):
    """Test faylingizni o'qiydi va anomaliyalarni topadi."""
    all_logs = []

    # JSONL faylini bir marta o'qib, barcha loglarni saqlaymiz
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            log_item = json.loads(line)
            all_logs.append(log_item)

    # Statistikalarni hisoblaymiz
    stats = compute_statistics(all_logs)
    print(
        f"📊 Statistika: mean_bytes={stats['mean_bytes']:.2f}, std_bytes={stats['std_bytes']:.2f}, "
        f"mean_failed={stats['mean_failed']:.4f}, std_failed={stats['std_failed']:.4f}\n"
    )

    anomalies = []
    total_logs = len(all_logs)

    # Har bir logni tahlil qilamiz
    for log_item in all_logs:
        result = analyze_log_item(log_item, stats)
        if result['is_anomaly']:
            anomalies.append(result)

    print(
        f"Jami o'qilgan loglar: {total_logs} ta | Topilgan anomaliyalar: {len(anomalies)} ta\n"
    )

    # Dastlabki 3 ta anomaliyani ekranga chiqaramiz
    for a in anomalies[:3]:
        print(
            f"🚨 User: {a['user_id']} | Risk Score: {a['risk_score']} | Log: {a['log']}"
        )

    return anomalies


# --- ISHLATISH ---
test_file_name = 'test_logs.jsonl'
found_anomalies = process_test_file(test_file_name)

📊 Statistika: mean_bytes=384627.81, std_bytes=3532229.86, mean_failed=0.0918, std_failed=0.2888

Jami o'qilgan loglar: 4432 ta | Topilgan anomaliyalar: 450 ta

🚨 User: user_0027 | Risk Score: 5.15 | Log: {'event_id': '00971ff8-9542-4daa-a28a-3abc45df4570', 'timestamp': '2026-08-01T00:01:35Z', 'user_id': 'user_0027', 'department': 'Engineering', 'role': 'admin', 'action': 'login', 'resource': '/api/v1/login', 'method': 'DELETE', 'status_code': 401, 'source_ip': '10.0.30.121', 'source_country': 'RU', 'source_city': 'Moscow', 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36', 'session_id': '74078792-60a', 'data_transferred_bytes': 1383}
🚨 User: user_0020 | Risk Score: 3.15 | Log: {'event_id': 'b7b0b0de-70aa-48c7-82c1-49647c1bb1bf', 'timestamp': '2026-08-01T03:43:44Z', 'user_id': 'user_0020', 'department': 'Finance', 'role': 'admin', 'action': 'outlook', 'resource': '/api/v1/users', 'method': 'PUT', 'status_code': 401, 'source_ip': 